In [1]:
import sys
import os

# Get the absolute path of the parent directory
parent_dir = os.path.abspath('..')

# Add it to sys.path so Python can find the 'src' module
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

print(f"Added {parent_dir} to Python path.")

Added /opt/spark to Python path.


**imports and Spark session**

In [2]:
from src.utils.spark_session import get_spark_session
from src.utils.s3_reader import read_delta_table

print("Imports successful")

Imports successful


In [4]:
spark = get_spark_session("SilverLayer-Orders")
print("Spark session ready")

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-36f804fa-071d-46c1-88ee-ef15555bb936;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.3.2 in central
	found io.delta#delta-storage;3.3.2 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
downloading https://repo1.maven.org/maven2/io/delta/delta-spark_2.12/3.3.2/delta-spark_2.12-3.3.2.jar ...
	[SUCCESSFUL ] io.delta#delta-spark_2.12;3.3.2!delta-spark_2.12.jar (1657ms)
downloading https://repo1.maven.org/maven2/io/delta/delta-storage/3.3.2/delta-storage-3.3.2.jar ...
	[SUCCESSFUL ] io.delta#delta-storage;3.3.2!delta-storage.jar (131ms)
downloading https://repo1.maven.org/maven2/org/antlr/antlr4-runtime/4.9.3/antlr4-runtime-4.9.3.jar ...
	[SUCCESSFUL ] org.antlr#antlr4-runtime;4.9.3!antlr4-runtime.jar (268ms)
:: resolution report :: resolve 4491ms :: artifacts dl

Spark session ready


**list Bronze folder using Spark**

In [13]:
from src.config.settings import BRONZE_PATH

# Create URI + Path
uri = spark._jvm.java.net.URI(BRONZE_PATH)
path = spark._jvm.org.apache.hadoop.fs.Path(BRONZE_PATH)

# Get correct filesystem based on s3a://
fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(
    uri,
    spark._jsc.hadoopConfiguration()
)

# List all items in bronze
files = fs.listStatus(path)

for file in files:
    print(file.getPath().toString())

s3a://olist-lakehouse-2026/bronze/holidays_dataset
s3a://olist-lakehouse-2026/bronze/olist_customers_dataset
s3a://olist-lakehouse-2026/bronze/olist_geolocation_dataset
s3a://olist-lakehouse-2026/bronze/olist_order_items_dataset
s3a://olist-lakehouse-2026/bronze/olist_order_payments_dataset
s3a://olist-lakehouse-2026/bronze/olist_order_reviews_dataset
s3a://olist-lakehouse-2026/bronze/olist_orders_dataset
s3a://olist-lakehouse-2026/bronze/olist_products_dataset
s3a://olist-lakehouse-2026/bronze/olist_sellers_dataset
s3a://olist-lakehouse-2026/bronze/product_category_name_translation


**read one Bronze dataset**

In [16]:
df_orders = read_delta_table(
    spark,
    "bronze",
    "olist_orders_dataset"
)

print("Bronze dataset loaded successfully")

Bronze dataset loaded successfully


**Preview sample rows**

In [17]:
df_orders.show(10, truncate=False)

[Stage 38:>                                                         (0 + 1) / 1]

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|995392413cee61cc1fcec9807870ea8c|4bf24904ec428325abad2791fcca1aff|delivered   |2017-09-04 22:24:05     |2017-09-04 22:43:54|2017-09-13 17:20:04         |2017-09-22 21:09:32          |2017-09-27 00:00:00          |
|b39de9ed2bb8fd08a970e4517c698824|ed18b557140ff674f36cbcf73921dbbe|delivered   |2018-03-27 09:15:17     |2018-03-27 09:30:12|2018-03-27 23:5

**Inspect schema**

In [8]:
df_orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



**Count all rows**

In [9]:
total_rows = df_orders.count()
print(f"Total rows: {total_rows}")

Total rows: 99441


**Check distinct business key**

In [10]:
distinct_orders = df_orders.select("order_id").distinct().count()
print(f"Distinct order_id: {distinct_orders}")

[Stage 16:==================================>                       (3 + 2) / 5]

Distinct order_id: 99441


**Quick null profile**

In [11]:
from pyspark.sql.functions import col, when, sum

df_orders.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_orders.columns
]).show()

[Stage 24:==============================================>           (4 + 1) / 5]

+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|       0|          0|           0|                       0|              160|                        1783|                         2965|                            0|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+



**inspect order statuses**

In [18]:
df_orders.groupBy("order_status").count().show(truncate=False)

[Stage 41:==============================================>           (4 + 1) / 5]

+------------+-----+
|order_status|count|
+------------+-----+
|shipped     |1107 |
|canceled    |625  |
|approved    |2    |
|invoiced    |314  |
|delivered   |96478|
|unavailable |609  |
|processing  |301  |
|created     |5    |
+------------+-----+



**save the cleaned dataframe**

In [19]:
df_orders_clean = df_orders

**Save to Silver layer**

In [20]:
from src.utils.s3_writer import write_delta_table

write_delta_table(
    df_orders_clean,
    "silver",
    "olist_orders_dataset_clean"
)

Saved successfully to s3a://olist-lakehouse-2026/silver/olist_orders_dataset_clean


**Important validation after save**

In [21]:
df_silver_test = read_delta_table(
    spark,
    "silver",
    "olist_orders_dataset_clean"
)

df_silver_test.show(5)
df_silver_test.printSchema()

[Stage 55:>                                                         (0 + 1) / 1]

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|  